# Ordered Logistic Regression Results: FAIRˆ2 Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIRˆ2](https://doi.org/10.71728/senscience.y7m0-f273) dataset—'Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya'—using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")
print(f"Description: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Spatial Coverage: {meta.spatialCoverage}")
print(f"Temporal Coverage: {meta.temporalCoverage}")
print(f"Fields with personal or sensitive information: {meta.personalSensitiveInformation}\n")
print(f"Authors: {[a['@id'] for a in meta.author] if hasattr(meta, 'author') else 'Not available'}")

## 2. Data Overview
List available record sets, and preview their fields by `@id` as defined by the Croissant schema.

In [ ]:
# List available record sets by @id
from mlcroissant.types import RecordSet

print("Available record sets:")
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    recordsets = dataset.metadata.recordSet
elif hasattr(dataset, 'record_sets') and dataset.record_sets:
    recordsets = dataset.record_sets
else:
    recordsets = [rs for rs in dataset.record_sets]

if not recordsets:
    recordsets = list(dataset.record_sets.keys())
if isinstance(recordsets, dict):
    recordsets = list(recordsets.keys())
if isinstance(recordsets[0], dict) and '@id' in recordsets[0]:
    recordset_ids = [rs['@id'] for rs in recordsets]
else:
    recordset_ids = [str(rs) for rs in recordsets]
for rid in recordset_ids:
    print(f"- {rid}")

# For each record set, show the available fields (by @id)
print("\nFields available by record set:")
for rid in recordset_ids:
    try:
        recset = dataset.get_record_set(rid)
        if recset and hasattr(recset, 'fields'):
            fields = recset.fields
            if isinstance(fields, dict):
                field_ids = list(fields.keys())
            else:
                field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else str(f) for f in fields]
            print(f"Record set {rid}: {field_ids}")
        else:
            print(f"Record set {rid}: [Field information not available]")
    except Exception as e:
        print(f"Record set {rid}: [Unable to retrieve fields] - {e}")

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis.

Reference all entities (record sets, fields, etc.) by their `@id`.

In [ ]:
# List record set @ids explicitly, since some Croissant schemas may use custom or generated URIs
# We'll use dataset.record_sets, which maps @id -> RecordSet object

record_set_ids = list(dataset.record_sets.keys())
print(f"All available record set @ids: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    
    # Each record is a dict keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records found for record set {record_set_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    # Show columns (field @ids) and first rows
    print(f"Fields (@id) in {record_set_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filter, normalize, group by key field, etc.

All field names and groupings must use the Croissant `@id` as their identifier.

In [ ]:
# For this section, pick a record set and a numeric field by `@id`
if not dataframes:
    print("No dataframes loaded. Skipping EDA section.")
else:
    # Pick the first available DataFrame
    selected_record_set = list(dataframes.keys())[0]
    df = dataframes[selected_record_set]
    print(f"Using record set: {selected_record_set}")
    
    # List numeric-like columns
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if not numeric_cols:
        # Attempt fallback: try to coerce fields to numeric
        numeric_candidates = []
        for col in df.columns:
            try:
                pd.to_numeric(df[col])
                numeric_candidates.append(col)
            except Exception:
                continue
        numeric_cols = numeric_candidates
    if not numeric_cols:
        print("No numeric fields detected in the selected record set.")
    else:
        # Choose the first numeric field (@id)
        numeric_field_id = numeric_cols[0]
        print(f"Numeric field selected (@id): {numeric_field_id}")
        # Coerce to numeric for EDA
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in {selected_record_set} where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized values for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by another field if present
        # Pick a candidate categorical field (besides the numeric field)
        group_candidates = [c for c in df.columns if c != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by field (@id): {group_field_id}")
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                print(grouped_df.head())
            else:
                print(f"Field {group_field_id} not found in filtered data.")
        else:
            print("No additional field available for grouping.")

## 5. Visualization
Use matplotlib or pandas plotting to visualize numeric distributions or relationships by field `@id`.

In [ ]:
import matplotlib.pyplot as plt

if 'filtered_df' in locals() and not filtered_df.empty:
    # Histogram for the chosen numeric field
    plt.figure(figsize=(8, 4))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Plot mean by group (if grouping was performed)
    if 'grouped_df' in locals():
        grouped_df.plot(kind='bar', figsize=(10, 5))
        plt.title(f'Mean {numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()
else:
    print("No filtered data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated loading and brief exploration of the FAIRˆ2 dataset using `mlcroissant`. By referencing all schema objects by their Croissant `@id`, you can reliably perform programmatic, reproducible analyses across diverse datasets. Further work could include richer analysis, model-building, or integration of domain metadata.
